# Model Initialization - Package

This notebook initializes one dynamic Modelica model that belongs to a package, from its auxiliary package.

It loads the package and its auxiliary from `models/`, simulates the auxiliary to compute the initialization values, applies them to a copy of the package, and saves the initialized package to `outputs/`.

In [ ]:
import logging
import os
import shutil
import sys

sys.path.append(os.path.abspath(".."))

logging.getLogger("OMPython").setLevel(logging.ERROR)

import matplotlib.pyplot as plt
import pandas as pd
from OMPython import ModelicaSystemOMC, OMCSessionLocal

from scripts.helpers import *

# --- Configuration ---

# 1. Root dynamic model to initialize (qualified package name)
MODEL = "Demo_IEEE14.IEEE14DisconnectLine"

# 2. Input and output directories
MODELS_DIR = os.path.abspath("models")
OUTPUT_DIR = os.path.abspath("outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 3. Derived package/model names and paths
SOURCE_PACKAGE = MODEL.split(".")[0]
MODEL_DIR = os.path.join(MODELS_DIR, SOURCE_PACKAGE)
PATHS = package_workflow_paths(MODEL, MODEL_DIR, OUTPUT_DIR)

MODELS_PKG_PATH = PATHS["source_package_file"]

# Auxiliary package (input, from BuildAux) lives under models/
AUX_PACKAGE = PATHS["aux_package"]
AUX_ROOT_MODEL = PATHS["aux_root_model"]
AUX_DIR = os.path.join(MODELS_DIR, AUX_PACKAGE)
AUX_PACKAGE_FILE = os.path.join(AUX_DIR, "package.mo")

# Initialized package (output) is written under outputs/
INITIALIZED_PACKAGE = PATHS["initialized_package"]
INITIALIZED_DIR = PATHS["initialized_dir"]
INITIALIZED_ROOT_MODEL = PATHS["initialized_root_model"]
INITIALIZED_PACKAGE_FILE = PATHS["initialized_package_file"]
INITIALIZED_ORDER_FILE = PATHS["initialized_order_file"]

# 4. Output directory for OpenModelica build artifacts
RUN_OUTPUT_DIR = os.path.join(OUTPUT_DIR, INITIALIZED_PACKAGE + "_outputs")
os.makedirs(RUN_OUTPUT_DIR, exist_ok=True)

# 5. Library paths
# Modelica Standard Library
OMLIB_DIR = os.path.join(os.path.expanduser("~"), ".openmodelica", "libraries")
os.environ["OPENMODELICALIBRARY"] = OMLIB_DIR
MODELICA_PKG_PATH = os.path.join(OMLIB_DIR, "Modelica 3.2.3+maint.om", "package.mo")
# Dynawo Modelica library, for now the one the Julia family downloads
DYNAWO_PKG_PATH = os.path.abspath(
    os.path.join("..", "..", "julia_openmodelica", "dynawo_library", "Dynawo", "package.mo")
)

# 6. INIT model selection for components with multiple INIT profiles (leave empty for default).
INIT_MODEL_BY_COMPONENT = {}

# 7. Variable to plot after the initialized simulation
PLOT_VARIABLE = "Load2.terminal.V.re"

## Load and Validate the Models

The dynamic package and its auxiliary are loaded and checked, and the user configuration is validated against the package root model, before any changes are made.

In [ ]:
# 1. Load the original dynamic package and validate the configuration against it
SourceOMC = OMCSessionLocal()
omc_call(SourceOMC, f'loadFile("{MODELICA_PKG_PATH}")')
omc_call(SourceOMC, "loadModel(Complex)")
omc_call(SourceOMC, "loadModel(ModelicaServices)")
omc_call(SourceOMC, f'loadFile("{DYNAWO_PKG_PATH}")')
omc_call(SourceOMC, f'loadFile("{MODELS_PKG_PATH}")')
config = check_user_configuration_package(
    SourceOMC,
    model=MODEL,
    init_model_by_component=INIT_MODEL_BY_COMPONENT,
)
chain = config["model_chain"]

# 2. Load the auxiliary package
AuxOMC = OMCSessionLocal()
omc_call(AuxOMC, f'loadFile("{MODELICA_PKG_PATH}")')
omc_call(AuxOMC, "loadModel(Complex)")
omc_call(AuxOMC, "loadModel(ModelicaServices)")
omc_call(AuxOMC, f'loadFile("{DYNAWO_PKG_PATH}")')
omc_call(AuxOMC, f'loadFile("{AUX_PACKAGE_FILE}")')
print("Checking the auxiliary root model...")
chk_aux = send_expression(AuxOMC, f"checkModel({AUX_ROOT_MODEL})", parsed=False)
print(chk_aux)

In [ ]:
INITIALIZED_NAME_MAP = initialized_name_map(chain, INITIALIZED_PACKAGE)

## Extract Initialization Values

The auxiliary package is simulated, and its outputs provide the initialization values to apply to each model in the inheritance chain.

In [ ]:
# Build and simulate the auxiliary root model.
AuxSystem = ModelicaSystemOMC(session=AuxOMC)
# The package is already loaded in this session; passing the file would copy only package.mo
AuxSystem.model(model_name=AUX_ROOT_MODEL)
AuxSystem.simulate(resultfile=AUX_PACKAGE + "_res.mat")

# Extract initialization values per original class in the inheritance chain.
initializable_by_model = {}
init_values_by_model = {}

for model in chain:
    components = get_all_components(SourceOMC, model)
    initializable_components = get_initializable_components(components, INIT_MODEL_BY_COMPONENT)
    init_values_by_component = extract_all_initialization_values(AuxSystem, components, INIT_MODEL_BY_COMPONENT)

    initializable_by_model[model] = initializable_components
    init_values_by_model[model] = init_values_by_component

## Build Initialized Package

A copy of each model in the package is populated with the extracted values and saved as the initialized package in `outputs/`.

In [ ]:
send_expression(SourceOMC, f"deleteClass({INITIALIZED_PACKAGE})")
omc_call(SourceOMC, f'loadString("within ; package {INITIALIZED_PACKAGE} end {INITIALIZED_PACKAGE};")', parsed=False)

for model in chain:
    initialized_model = INITIALIZED_NAME_MAP[model]
    initialized_name = initialized_model.split(".")[-1]

    print("Copying and initializing", model, "->", initialized_model)
    omc_call(SourceOMC, f'copyClass({model}, "{initialized_name}", {INITIALIZED_PACKAGE})')

    apply_initialization_modifiers(
        SourceOMC,
        initialized_model,
        initializable_by_model[model],
        init_values_by_model[model],
        INIT_MODEL_BY_COMPONENT,
    )

if os.path.isdir(INITIALIZED_DIR):
    shutil.rmtree(INITIALIZED_DIR)
os.makedirs(INITIALIZED_DIR, exist_ok=True)

write_package_files(
    INITIALIZED_PACKAGE_FILE,
    INITIALIZED_ORDER_FILE,
    INITIALIZED_PACKAGE,
    package_class_names(chain, INITIALIZED_NAME_MAP),
)

save_initialized_package_classes(
    SourceOMC,
    chain,
    INITIALIZED_NAME_MAP,
    INITIALIZED_DIR,
)

# Load the generated initialized package into a fresh session and validate it
InitializedOMC = OMCSessionLocal()
omc_call(InitializedOMC, f'loadFile("{MODELICA_PKG_PATH}")')
omc_call(InitializedOMC, "loadModel(Complex)")
omc_call(InitializedOMC, "loadModel(ModelicaServices)")
omc_call(InitializedOMC, f'loadFile("{DYNAWO_PKG_PATH}")')
omc_call(InitializedOMC, f'loadFile("{INITIALIZED_PACKAGE_FILE}")')
print("Checking the initialized root model...")
chk_initialized = send_expression(InitializedOMC, f"checkModel({INITIALIZED_ROOT_MODEL})", parsed=False)
print(chk_initialized)

## Validate Initialized Package

The initialized package is loaded independently and simulated to verify that the final output can run.

In [ ]:
InitializedSystem = ModelicaSystemOMC(session=InitializedOMC, work_directory=RUN_OUTPUT_DIR)
InitializedSystem.model(model_name=INITIALIZED_ROOT_MODEL)

simflags = simulation_flags_without_log_stats(InitializedOMC, INITIALIZED_ROOT_MODEL)
initialized_resultfile_prefix = INITIALIZED_PACKAGE

sim_result = send_expression(
    InitializedOMC,
    f'simulate({INITIALIZED_ROOT_MODEL}, outputFormat="csv", '
    f'fileNamePrefix="{initialized_resultfile_prefix}", simflags="{simflags}")',
    parsed=False,
)

print(sim_result)

initialized_resultfile = os.path.join(
    str(InitializedSystem.getWorkDirectory()),
    initialized_resultfile_prefix + "_res.csv",
)

In [ ]:
result_columns = pd.read_csv(initialized_resultfile, nrows=0).columns
if PLOT_VARIABLE not in result_columns:
    raise RuntimeError(f'PLOT_VARIABLE "{PLOT_VARIABLE}" is not a variable in the simulation result.')
initialized_df = pd.read_csv(initialized_resultfile, usecols=["time", PLOT_VARIABLE])

fig, ax = plt.subplots()
ax.plot(initialized_df["time"], initialized_df[PLOT_VARIABLE], label=PLOT_VARIABLE)
ax.legend(loc="lower right")
ax.set_title("Initialized package response")
ax.set_xlabel("Time (s)")
ax.set_ylabel(PLOT_VARIABLE)